## Importações

In [15]:
import pandas as pd
import functions
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from requests.exceptions import ReadTimeout
import discogs_client
from time import sleep
import json

## Criando sessão pyspark

In [16]:
import os
os.environ['SPARK_HOME'] = '/home/david/Documentos/UFABC/PGC/Codigos/code/Spark'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'jupyter' 
os.environ['PYSPARK_DRIVER_OPTS'] = 'notebook'
os.environ['PYSPARK_PYTHON'] = 'python'

In [17]:
from pyspark.sql import SparkSession

In [18]:
spark = SparkSession.builder \
        .appName('PGC') \
        .getOrCreate()

24/08/05 20:13:53 WARN Utils: Your hostname, david-Nitro resolves to a loopback address: 127.0.1.1; using 192.168.15.5 instead (on interface wlp9s0)
24/08/05 20:13:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/05 20:13:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [19]:
spark.read.csv(
                                    './code/data/raw/lastfm/lastfm-dataset-360K/usersha1-artmbid-artname-plays.tsv',
                                    sep='\t',
                                    inferSchema=True,
                                    encoding='utf-8'
                                )

24/08/05 20:14:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


DataFrame[_c0: string, _c1: string, _c2: string, _c3: double]

## Spotify API

In [16]:
def get_genre(artistName):
    try:
        print(f'Artista: {artistName}')
        birdy_uri = 'spotify:artist:2WX2uTcsvV5OnS0inACecP'
        sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials(), requests_timeout=10, retries=10)
        
        result = sp.search(artistName)
        track = result['tracks']['items'][0]
        
        artist = sp.artist(track["artists"][0]["external_urls"]["spotify"])
        
        album = sp.album(track["album"]["external_urls"]["spotify"])
    
        print(f'Gente: {artist["genres"]}')
        if len(artist["genres"]) != 0:
            return artist["genres"]
        else:
            return album["genres"]
    except ReadTimeout:
        print('Timeout')

# Discogs API

In [17]:
d = discogs_client.Client('ExampleApplication/0.1',user_token='mDokHVmBVpXhrdfBYEJCJUgxEUipNOQhioIBNpvh')

def get_artist_genre_discogs(artist):
    results = d.search(artist, type='release')

    if len(results) > 0:
        id = results[0].id
        return d.release(id).genres
    else:
        print(f'Artista não encontrado: {artist}')
        return []

## Datasets utilizados

### Yahoo Movies

In [5]:
path_yahoo_movies = 'Datasets/Yahoo Movies/dataset/ydata-ymovies-user-movie-ratings-train-v1_0.txt'
df_yahoo_movies = pd.read_csv(path_yahoo_movies, sep='\t', names=['user_id', 'movie_id', 'rating', 'converted_rating'])

In [6]:
df

,user_id,movie_id,rating,converted_rating
0,1,1800029049,12,5
1,1,1804857429,8,4
2,1,1800030906,13,5
3,1,1800018548,11,5
4,1,1800256362,9,4
...,...,...,...,...
211226,7642,1808405417,11,5
211227,7642,1807839027,8,4
211228,7642,1808405428,11,5
211229,7642,1808429384,10,4


In [12]:
print(functions.df_to_latex(df.head(), 'Primeiras 5 linhas do conjunto de dados Yahoo! Movies'))

\begin{table}[]
\centering
        \begin{tabular}{cccc}
\toprule
user_id & movie_id & rating & converted_rating \\
\midrule
1 & 1800029049 & 12 & 5 \\
1 & 1804857429 & 8 & 4 \\
1 & 1800030906 & 13 & 5 \\
1 & 1800018548 & 11 & 5 \\
1 & 1800256362 & 9 & 4 \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de dados Yahoo! Movies}
\end{table}
    


In [46]:
df_desc = pd.read_csv('../Datasets/Yahoo Movies/dataset/content-desc/movie_db_yoda', 
                      sep='\t', 
                      encoding='iso-8859-1', 
                      usecols = range(11),
                      names=['movie_id', 
                             'title', 
                             'synopsis', 
                             'running_time', 
                             'MPAA_rating', 
                             'reasons_for_the_MPAA_rating', 
                             'release_date_(yyyymmdd)',
                             'distributor',
                             'url_poster',
                             'pass',
                            'genres'])

In [47]:
df_desc = df_desc[['movie_id', 'title', 'genres']] 

In [51]:
df_desc = df_desc[df_desc['genres'] != r'\N']

In [52]:
print(functions.df_to_latex(df_desc.head(), 'Primeiras 5 linhas do conjunto de dados de descrição de conteúdo do Yahoo! Movies'))

\begin{table}[]
\centering
        \begin{tabular}{ccc}
\toprule
movie_id & title & genres \\
\midrule
1800010969 & The 1985 Admiral's Cup (1997) & Special Interest \\
1800011786 & 984 - Prisoner of the Future (1984) & Science Fiction/Fantasy \\
1800011850 & A's All-Star Almanac (1987) & Special Interest \\
1800012991 & The Adventures of Annie Oakley (1953) & Western \\
1800013061 & The Adventures of Black Beauty 1 (1972) & Kids/Family \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de dados de descrição de conteúdo do Yahoo! Movies}
\end{table}
    


### Yahoo Music

In [3]:
df2 = pd.read_csv('code/data/raw/yahoo_music/dataset/ydata-ymusic-user-artist-ratings-v1_0.txt', 
                  sep='\t', 
                  names=['userId', 'artistId ', 'rating'])

In [4]:
df2.shape

(115579440, 3)

In [6]:
df2_names = pd.read_csv('code/data/raw/yahoo_music/dataset/ydata-ymusic-artist-names-v1_0.txt', 
                        sep='\t', 
                        names=['artistId', 'artistName'], 
                        usecols=range(2), 
                        encoding='iso-8859-1')

In [7]:
df2_names.shape

(97956, 2)

In [5]:
df2_names['artistName'] = df2_names['artistName'].str.replace('"', '') 
df2_names['artistName'] = df2_names['artistName'].str.replace("'", '')

In [6]:
test = pd.read_csv('../Datasets/Diversos/genres_music/archive/test.csv')
train = pd.read_csv('../Datasets/Diversos/genres_music/archive/train.csv')

In [7]:
train = train[['Artist Name', 'Class']].drop_duplicates()

In [8]:
df2_names = df2_names[df2_names['artistId']>0]
df2_names['artistName'] = df2_names['artistName'].str.lower()

#### Tentando recuperar gêneros com a Discogs

Importando dicionário previamente populado

In [5]:
json_path = 'Datasets/Yahoo Music/yahoo_music_genres.json'

with open(json_path) as json_file:
    artists_genres = json.load(json_file)
    

In [18]:
genres_mod = {'artistId': list(artists_genres.keys()), 
              'artistName': [v[0] for _, v in artists_genres.items()],
              'artistGenre': [v[1] for _, v in artists_genres.items()] 
             }

In [21]:
df_artists_gen = pd.DataFrame.from_dict(genres_mod)

In [29]:
df_artists_gen

,artistId,artistName,artistGenre
0,1000001,bobby o,"[Hip Hop, Funk / Soul]"
1,1000002,jimmy z,"[Hip Hop, Rock]"
2,1000003,68 comeback,[Rock]
3,1000004,til tuesday,"[Rock, Pop]"
4,1000005,the (ec) nudes,"[Jazz, Rock]"
...,...,...,...
97949,1101110,14 karat soul,[Funk / Soul]
97950,1101111,the relativez,[Hip Hop]
97951,1101112,crooked i,[Rock]
97952,1101113,skg,[Electronic]


In [15]:
artists_dict = df2_names.set_index('artistId').to_dict()['artistName']
max_key = max(map(int, artists_genres.keys()))
artists_dict = {k: v for k, v in artists_dict.items() if int(k) > max_key}


for id, artist in artists_dict.items():
    print(len(artists_genres))
    genre = get_artist_genre_discogs(artist)
    print(f'{artist}: {genre}')
    artists_genres[id] = (artist, genre)
    sleep(1)

1100416

In [28]:
artists_genres

{'1000001': ['bobby o', ['Hip Hop', 'Funk / Soul']],
 '1000002': ['jimmy z', ['Hip Hop', 'Rock']],
 '1000003': ['68 comeback', ['Rock']],
 '1000004': ['til tuesday', ['Rock', 'Pop']],
 '1000005': ['the (ec) nudes', ['Jazz', 'Rock']],
 '1000006': ['.38 special', ['Rock']],
 '1000008': ['1 + 1', ['Rock']],
 '1000009': ['1 of the girls', ['Hip Hop', 'Funk / Soul']],
 '1000010': ['1,000 clowns', ['Hip Hop']],
 '1000011': ['10 k.a.n.s', ['Hip Hop']],
 '1000012': ['10,000 maniacs', ['Rock']],
 '1000013': ['100 degrees celsius', ['Electronic']],
 '1000014': ['100 flowers', ['Rock']],
 '1000015': ['100 proof', ['Funk / Soul']],
 '1000016': ['1000 homo djs', ['Electronic']],
 '1000017': ['1000 mona lisas', ['Rock']],
 '1000018': ['101', ['Electronic', 'Rock', 'Pop']],
 '1000019': ['101 jade 4 u', ['Electronic']],
 '1000020': ['101 north', ['Jazz', 'Funk / Soul']],
 '1000021': ['101 strings', ['Pop']],
 '1000022': ['108', ['Electronic']],
 '1000023': ['10cc', ['Rock']],
 '1000024': ['10db', ['El

In [41]:
max(map(int, artists_genres.keys())) - max_key

16649

#### Salvando o JSON em disco

In [20]:
with open(json_path, 'w') as fp:
    json.dump(artists_genres, fp)

In [22]:
df2_names

,artistId,artistName
2,1000001,bobby o
3,1000002,jimmy z
4,1000003,68 comeback
5,1000004,til tuesday
6,1000005,the (ec) nudes
...,...,...
97951,1101110,14 karat soul
97952,1101111,the relativez
97953,1101112,crooked i
97954,1101113,skg


In [35]:
df_gen = pd.DataFrame(artists_genres.items(), columns=['artistId', 'artistName']) 

In [33]:
df_gen[['valor1', 'nested_list']] = pd.DataFrame(df_gen['col1'].tolist(), index=df.index)

AttributeError: 'list' object has no attribute 'get'

In [36]:
print(functions.df_to_latex(df2_names.head(), 'Primeiras 5 linhas do conjunto de nomes de artistas do Yahoo! Music'))

\begin{table}[]
\centering
        \begin{tabular}{cc}
\toprule
artistId & artistName \\
\midrule
-100 & Not Applicable \\
-99 & Unknown Artist \\
1000001 & Bobby "O" \\
1000002 & Jimmy "Z" \\
1000003 & '68 Comeback \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de nomes de artistas do Yahoo! Music}
\end{table}
    


In [15]:
print(functions.df_to_latex(df2.head(), 'Primeiras 5 linhas do conjunto de dados Yahoo! Music'))

\begin{table}[]
\centering
        \begin{tabular}{ccc}
\toprule
userId & artistId  & rating \\
\midrule
1 & 1000125 & 90 \\
1 & 1006373 & 100 \\
1 & 1006978 & 90 \\
1 & 1007035 & 100 \\
1 & 1007098 & 100 \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de dados Yahoo! Music}
\end{table}
    


-------------------------------

### LastFm

In [5]:
import polars as pl
import pandas as pd
import chardet

In [10]:
    def _detect_encoding(file_path):
        with open(file_path, 'rb') as file:
            raw_data = file.read()
            result = chardet.detect(raw_data)
            return result['encoding']
        

In [23]:
_detect_encoding('/code/data/raw/lastfm/usersha1-artmbid-artname-plays.tsv')

FileNotFoundError: [Errno 2] No such file or directory: '/code/data/raw/lastfm/usersha1-artmbid-artname-plays.tsv'

In [13]:
lastfm = pl.read_csv('code/data/raw/lastfm/lastfm-dataset-360K/usersha1-artmbid-artname-plays.tsv', 
                    separator = '\t',
                    encoding = 'utf-8',
                    new_columns=['userId', 'artistId', 'artistName', 'plays'],
                    has_header = False)
                    #lineterminator = '\n',
                    #nrows = 17559530)

ComputeError: could not parse `"ÂÈÎ-66", İñòğàäíûé îğêåñòğ, Àëåêñåé Ìàæóêîâ` as dtype `str` at column 'column_3' (column number 3)

The current offset in the file is 437770335 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `dtypes` argument
- setting `ignore_errors` to `True`,
- adding `"ÂÈÎ-66", İñòğàäíûé îğêåñòğ, Àëåêñåé Ìàæóêîâ` to the `null_values` list.

Original error: ```string field is not properly escaped```

In [27]:
df_lastfm = spark.read.csv('code/data/raw/lastfm/lastfm-dataset-360K/usersha1-artmbid-artname-plays.tsv', 
                           sep=r'\t', 
                           inferSchema = True)

In [29]:
columns = ['userId', 'artistId', 'artistName', 'plays']
df_lastfm = df_lastfm.toDF(*columns)

In [35]:
df_lastfm.count()

17559530

In [36]:
df_artists = df_lastfm.select('artistId', 'artistName')

In [38]:
df_artists = df_artists.dropDuplicates()

In [39]:
df_artists.count()

295257

In [42]:
df_artists = df_artists.coalesce(1)
df_artists.write.csv('lastfm_artists.csv', header=True)

#### Obtendo gêneros

Obtendo dicionário com gêneros previamente salvos

In [29]:
json_path = 'Datasets/lastfm/lastfm_music_genres.json'

try:
    with open(json_path) as json_file:
        artists_genres = json.load(json_file)
except:
    artists_genres = {}

In [3]:
lastfm_artists = lastfm[['artistId', 'artistName']].drop_duplicates()

In [31]:
lastfm_artists.shape

(295262, 2)

In [4]:
lastfm
artists_dict = lastfm_artists.set_index('artistId').to_dict()['artistName']

In [6]:
with open('lastfm_artists.json', 'w') as fp:
    json.dump(artists_dict, fp)

In [34]:
if len(artists_genres) > 0:
    max_key = max(map(int, artists_genres.keys()))
else:
    max_key = -1
    
# artists_dict = {k: v for k, v in artists_dict.items() if int(k) > max_key}


for id, artist in artists_dict.items():
    print(len(artists_genres))
    genre = get_artist_genre_discogs(artist)
    print(f'{artist}: {genre}')
    artists_genres[id] = (artist, genre)
    sleep(1)

0
betty blowtorch: ['Rock']
1
die ärzte: ['Rock']
2
melissa etheridge: ['Rock']
3
elvenking: ['Rock']
4
juliette lewis and the licks: ['Rock']
5
red hot chili pippers: ['Rock']
6
magica: ['Electronic']
7
the black dahlia murder: ['Rock']
8
murmurs: ['Rock']
9
lunachicks: ['Rock']
10
walls of jericho: ['Rock']
11
letzte instanz: ['Non-Music', 'Pop', 'Folk, World, & Country']
12
goldfrapp: ['Electronic', 'Non-Music', 'Pop']
13
horrorpops: ['Rock']
14
butchies, the: ['Rock']
15
jack off jill: ['Rock']
16
babes in toyland: ['Rock']
17
Artista não encontrado: dropkick murfhys
dropkick murfhys: []
18
all my faults: ['Funk / Soul']
19
le tigre: ['Electronic']
20
schandmaul: ['Rock', "Children's"]
21
edguy: ['Rock']
22
Artista não encontrado: maximum the harmone
maximum the harmone: []
23
all ends: ['Funk / Soul']
24
johnson jack: ['Jazz']
25
eluveitie: ['Rock']
26
rasputina: ['Rock']
27
Artista não encontrado: london afrer midnight
london afrer midnight: []
28
the killers: ['Rock']
29
夢中夢: ['

KeyboardInterrupt: 

Salvando json em disco

In [35]:
with open(json_path, 'w') as fp:
    json.dump(artists_genres, fp)

----------------------------------------------

## Netflix Prize

In [19]:
column_names = ['MovieID', 'YearOfRelease', 'Title']
movie_titles = pd.read_csv('../Datasets/Netflix/movie_titles.csv', 
                           encoding='ISO-8859-1', 
                           header=None,
                           names=column_names,
                           on_bad_lines ='skip')

In [20]:
movie_titles

,MovieID,YearOfRelease,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
...,...,...,...
17429,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17430,17767,2004.0,Fidel Castro: American Experience
17431,17768,2000.0,Epoch
17432,17769,2003.0,The Company


In [21]:
print(functions.df_to_latex(movie_titles.head(), 'Primeiras 5 linhas do conjunto de dados Netflix Prize'))

\begin{table}[]
\centering
        \begin{tabular}{ccc}
\toprule
MovieID & YearOfRelease & Title \\
\midrule
1 & 2003.000000 & Dinosaur Planet \\
2 & 2004.000000 & Isle of Man TT 2004 Review \\
3 & 1997.000000 & Character \\
4 & 1994.000000 & Paula Abdul's Get Up & Dance \\
5 & 2004.000000 & The Rise and Fall of ECW \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de dados Netflix Prize}
\end{table}
    


In [14]:
# FONTE: https://www.kaggle.com/code/nityaverma19/user-based-cf

def read_combined_data(file_path):
    # Initialize lists to store data
    movie_ids = []
    customer_ids = []
    ratings = []
    dates = []

    # Read the file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Initialize variables
    current_movie_id = None

    # Process each line
    for line in lines:
        # Strip leading/trailing whitespaces
        line = line.strip()

        # Check for movie ID
        if line.endswith(':'):
            # Update current movie ID
            current_movie_id = line[:-1].strip()
        elif line:  # Check if the line is not empty
            # Extract customer data
            data = line.strip().split(',')
            if len(data) >= 3:
                customer_id = data[0].strip()
                rating_str = data[1].strip()
                date = data[2].strip()

                try:
                    rating = float(rating_str)
                except ValueError:
                    print(f"Skipping invalid rating (not a valid float): '{rating_str}'")
                    continue

                # Append data to lists
                movie_ids.append(current_movie_id)
                customer_ids.append(customer_id)
                ratings.append(rating)
                dates.append(date)
            else:
                print(f"Skipping line with insufficient data: '{line}'")

    # Create DataFrame
    combined_data = pd.DataFrame({
        'MovieID': movie_ids,
        'CustomerID': customer_ids,
        'Rating': ratings,
        'Date': dates
    })

    return combined_data

# Read combined_data_1.txt
file_path = '../Datasets/Netflix/combined_data_1.txt'
print(f"Reading file: {file_path}")
data_1 = read_combined_data(file_path)

# Display the first few rows of the DataFrame
print(data_1.head())

Reading file: ../Datasets/Netflix/combined_data_1.txt
  MovieID CustomerID  Rating        Date
0       1    1488844     3.0  2005-09-06
1       1     822109     5.0  2005-05-13
2       1     885013     4.0  2005-10-19
3       1      30878     4.0  2005-12-26
4       1     823519     3.0  2004-05-03


In [15]:
data_1

,MovieID,CustomerID,Rating,Date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03
...,...,...,...,...
24053759,4499,2591364,2.0,2005-02-16
24053760,4499,1791000,2.0,2005-02-10
24053761,4499,512536,5.0,2005-07-27
24053762,4499,988963,3.0,2005-12-20


In [16]:
print(functions.df_to_latex(data_1.head(), 'Primeiras 5 linhas do conjunto de dados Netflix Prize'))

\begin{table}[]
\centering
        \begin{tabular}{cccc}
\toprule
MovieID & CustomerID & Rating & Date \\
\midrule
1 & 1488844 & 3.000000 & 2005-09-06 \\
1 & 822109 & 5.000000 & 2005-05-13 \\
1 & 885013 & 4.000000 & 2005-10-19 \\
1 & 30878 & 4.000000 & 2005-12-26 \\
1 & 823519 & 3.000000 & 2004-05-03 \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de dados Netflix Prize}
\end{table}
    


## MovieLens

In [3]:
df = pd.read_csv('../Datasets/MovieLens/100K/ml-100k/u.data',
                names=['userId', 'itemId', 'rating', 'timestamp'],
                sep='\t')

In [5]:
df = df[['userId', 'itemId', 'rating']]

In [7]:
print(functions.df_to_latex(df.head(), 'Primeiras 5 linhas do conjunto de dados MovieLens 20M'))

\begin{table}[]
\centering
        \begin{tabular}{ccc}
\toprule
userId & itemId & rating \\
\midrule
196 & 242 & 3 \\
186 & 302 & 3 \\
22 & 377 & 1 \\
244 & 51 & 2 \\
166 & 346 & 1 \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de dados MovieLens 20M}
\end{table}
    


In [13]:
df = pd.read_csv('../Datasets/MovieLens/100K/ml-100k/u.item',
                 usecols=range(11),
                 names = ['movieId', 
                       'movieTitle', 
                       'releaseDate', 
                       'videoReleaseDate', 
                       'imdbURL', 
                       'unknow', 
                       'action', 
                       'adventure', 
                       'animation',
                       'Children',
                       'documentary'],
                sep='|',
                encoding='iso-8859-1')

In [15]:
df = df[['movieId', 'movieTitle', 'action', 'adventure', 'animation']]

In [17]:
df['...'] = '...'

/tmp/ipykernel_3575/218817001.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['...'] = '...'


In [19]:
print(functions.df_to_latex(df.head(), 'Primeiras 5 linhas do conjunto de descrições de itens do MovieLens 20M'))

\begin{table}[]
\centering
        \begin{tabular}{cccccc}
\toprule
movieId & movieTitle & action & adventure & animation & ... \\
\midrule
1 & Toy Story (1995) & 0 & 0 & 1 & ... \\
2 & GoldenEye (1995) & 1 & 1 & 0 & ... \\
3 & Four Rooms (1995) & 0 & 0 & 0 & ... \\
4 & Get Shorty (1995) & 1 & 0 & 0 & ... \\
5 & Copycat (1995) & 0 & 0 & 0 & ... \\
\bottomrule
\end{tabular}

\caption{Primeiras 5 linhas do conjunto de descrições de itens do MovieLens 20M}
\end{table}
    


# Exemplo de utility matrix

In [4]:
data = [['usuario1', 2, '-', 5, '-'], ['usuario2', '-', 1 , '-', 4], ['usuario3', 3, 1, '-', 2], ['usuario4', '-', 2, '-', 5]]

In [5]:
df_ex = pd.DataFrame(data, columns=['Usuarios', 'Item 1', 'Item 2', 'Item 3', 'Item 4'])

In [7]:
print(functions.df_to_latex(df_ex, 'Exemplo de matriz de utilidade'))

\begin{table}[]
\centering
        \begin{tabular}{ccccc}
\toprule
Usuarios & Item 1 & Item 2 & Item 3 & Item 4 \\
\midrule
usuario1 &      2 &      - &      5 &      - \\
usuario2 &      - &      1 &      - &      4 \\
usuario3 &      3 &      1 &      - &      2 \\
usuario4 &      - &      2 &      - &      5 \\
\bottomrule
\end{tabular}

\caption{Exemplo de matriz de utilidade}
\end{table}
    


C:\Users\David\Documents\UFABC\PGC\Codigos\functions.py:5: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = latex + df.to_latex(index=False, header=True,


In [5]:
print(df_ex.to_latex(index=False, header=True, column_format='ccccc'))

\begin{tabular}{ccccc}
\toprule
Usuarios & Item 1 & Item 2 & Item 3 & Item 4 \\
\midrule
usuario1 &      3 &      - &      2 &      - \\
usuario2 &      - &      4 &      - &      2 \\
usuario3 &      4 &      - &      3 &      - \\
usuario4 &      - &      5 &      4 &      1 \\
\bottomrule
\end{tabular}



C:\Users\David\AppData\Local\Temp\ipykernel_15908\372530190.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(df_ex.to_latex(index=False, header=True, column_format='ccccc'))


In [9]:
user,item,title = 'userId','movieId','title'

In [12]:
df = pd.read_csv('../Datasets/MovieLens/100K/ml-100k/u.data', sep='\t', header=None, names=[user,item,'rating','timestamp'])[[user,item,'rating']]

In [23]:
df = df.sort_values(by=['userId', 'movieId'])

In [24]:
print(functions.df_to_latex(df.head(), 'Exemplo de matriz de utilidade'))

\begin{table}[]
\centering
        \begin{tabular}{ccc}
\toprule
 userId &  movieId &  rating \\
\midrule
      1 &        1 &       5 \\
      1 &        2 &       3 \\
      1 &        3 &       4 \\
      1 &        4 &       3 \\
      1 &        5 &       3 \\
\bottomrule
\end{tabular}

\caption{Exemplo de matriz de utilidade}
\end{table}
    


C:\Users\David\Documents\UFABC\PGC\Codigos\functions.py:5: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex = latex + df.to_latex(index=False, header=True,


In [3]:
df = pd.read_csv('./Datasets/movie_titles.csv', encoding = 'ANSI', sep=',', usecols=range(3), lineterminator='\n', names=['movieId', 'year', 'title'])

In [ ]:
df

In [8]:
df2 = pd.read_csv('../Datasets/Netflix/netflix_genres.csv')

In [9]:
df2

,movieId,genres
0,1,Documentary|Animation|Family
1,3,Crime|Drama|Mystery
2,4,Family
3,5,Documentary|Sport
4,6,Documentary
...,...,...
12274,17764,Comedy|Drama|History|Romance
12275,17765,Action|Adventure|Family|Sci-Fi
12276,17768,Action|Drama|Fantasy
12277,17769,Drama|Music|Romance


In [ ]:
df_merge = pd.merge(df, df2, how='left')

In [ ]:
df_merge

In [ ]:
df_merge = df_merge[~df_merge['genres'].isna()]

In [ ]:
df_merge['genres'] = df_merge['genres'].astype(str)

In [ ]:
df_merge['n_bars'] = df_merge['genres'].str.count(r'[|]')

In [ ]:
df_merge[df_merge['n_bars'] > 9]

In [ ]:
df_merge[['genre1', 'genre2', 'genre3', 'genre4']] = 

# LastFm

In [ ]:
df = pd.read_csv()